# ETIS transformer comparison - frozen 13 October rule, phase 1

Fine-tune `UBC-NLP/MARBERTv2` and `CAMeL-Lab/bert-base-arabic-camelbert-da` with seeds **42, 43, 44**.
Use only the private, hash-verified development bundle. **No sealed test rows. No selection or deployment.**
Each seed trains a full model and two source-held-out models. Held sources are
`kaggle_arabic_phishing_fully_dataset` and `salloum_gaber_bilingual_corpus`.
LOSO training excludes held-source groups; thresholds retain at least 95% phishing recall on
other-source eligible validation. Dialectal validation is excluded and never moved into training.
Outputs include exact row identities, eligible-validation and LOSO predictions, 12 existing
invented everyday-probe scores, and a separate CPU latency pass. Local import replays the
probe scores using the existing Decision-v2 policy (high = eligible-validation threshold, low = 0.2).
Translated data and invented probes do not establish authentic Egyptian SMS performance.

Read the bundled README. The owner authorized this private Colab upload; do not upload the full
repository or dataset. No embedded URL is visited. The fixed training recipe is three epochs,
2e-5 learning rate, effective batch size 16, max 256 tokens per normalized 150-word overlapping
window (stride 75), max score across windows. Held rows/probes never affect training or threshold tuning.


In [ ]:
%pip install -q "transformers==4.45.2" "accelerate==0.34.2" "safetensors>=0.4.5,<1" "numpy>=1.26,<3"


In [ ]:
import hashlib, json, os, sys, zipfile
from pathlib import Path
from google.colab import files
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['WANDB_DISABLED'] = 'true'
BASE = Path('/content/etis-text')
BUNDLE = BASE / 'input'
OUT = BASE / 'returned'
BUNDLE.mkdir(parents=True, exist_ok=True)
uploaded = files.upload()  # Select only colab-bundle/etis-text-input.zip
if set(uploaded) != {'etis-text-input.zip'}:
    raise ValueError('Upload only etis-text-input.zip')
import io
with zipfile.ZipFile(io.BytesIO(uploaded['etis-text-input.zip'])) as z:
    names = z.namelist()
    if len(names) != len(set(names)) or any(Path(n).name != n or '/' in n or '\\' in n for n in names):
        raise ValueError('Unsafe or duplicate archive paths')
    manifest = json.loads(z.read('manifest.json'))
    if set(names) != set(manifest['files']) | {'manifest.json'}:
        raise ValueError('Input ZIP file set differs')
    for name, digest in manifest['files'].items():
        if hashlib.sha256(z.read(name)).hexdigest() != digest:
            raise ValueError('SHA-256 mismatch: ' + name)
    for name in names:
        (BUNDLE / name).write_bytes(z.read(name))
print('Manifest SHA-256:', hashlib.sha256((BUNDLE / 'manifest.json').read_bytes()).hexdigest())
print('Compare this value with local colab-bundle/checksums.json before continuing.')
del uploaded


In [ ]:
sys.path.insert(0, str(BUNDLE))
from transformer_contract import load_bundle, validate_run, read_json, sha, require, MODELS as ALL_MODELS, SEEDS as ALL_SEEDS
manifest, rows, rule, probes = load_bundle(BUNDLE)
import torch
require(torch.cuda.is_available(), 'Choose Runtime > Change runtime type > GPU')
print('GPU:', torch.cuda.get_device_name(0))
print('Verified input rows:', manifest['row_counts'], 'probes:', len(probes), 'sealed tests: 0')
from transformer_colab import run, package_results


## Optional: resume completed seeds from a previous session
Skip this cell for the first run. Upload your most recent partial results ZIP to retain completed
runs. Hashes and row sets are checked before copying any result. Never edit prediction files.


In [ ]:
resume = files.upload()  # Optional: select your previous etis-text-results.zip
require(set(resume) == {'etis-text-results.zip'}, 'Select only etis-text-results.zip')
with zipfile.ZipFile(io.BytesIO(resume['etis-text-results.zip'])) as z:
    names = z.namelist()
    receipt = json.loads(z.read('results_manifest.json'))
    require(len(names) == len(set(names)) and set(names) == set(receipt['files']) | {'results_manifest.json'}, 'Result ZIP file set mismatch')
    require(receipt['bundle_manifest_sha256'] == sha(BUNDLE / 'manifest.json'), 'Different input bundle')
    checked = {}
    for name, digest in receipt['files'].items():
        matches = [(m, s) for m in ALL_MODELS for s in ALL_SEEDS if name == f'{m}_seed{s}.json']
        require(len(matches) == 1, 'Unexpected result filename')
        raw = z.read(name)
        require(hashlib.sha256(raw).hexdigest() == digest, 'Returned file hash mismatch')
        m, s = matches[0]
        validate_run(json.loads(raw), BUNDLE, m, s)
        checked[name] = raw
    OUT.mkdir(parents=True, exist_ok=True)
    for name, raw in checked.items():
        if (OUT / name).exists():
            require((OUT / name).read_bytes() == raw, 'Conflicting completed result')
        (OUT / name).write_bytes(raw)
print('Restored completed runs:', len(checked))
del resume


## Fine-tune and collect development evidence
Run this cell after setup; the resume cell is optional. Default: both models, all three seeds.
For limited free GPU sessions, use e.g. `MODELS = ['marbertv2']; SEEDS = [42]`, download results,
then resume with the remaining jobs. Completed seeds are verified and skipped. There are 18
fine-tunes in total; free Colab may require multiple sessions. Each completed seed refreshes the
results ZIP. CPU latency is measured separately after GPU inference, including all preprocessing
and windows. It records hardware, two Torch threads, three warmups and 227 per-message timings.


In [ ]:
MODELS = ['marbertv2', 'camelbert_da']
SEEDS = [42, 43, 44]
archive = run(BUNDLE, OUT, models=MODELS, seeds=SEEDS)
print('Saved:', archive)


## Download and stop
Download after every session. The complete archive contains six JSON runs plus a SHA-256
results manifest. Extract into `evaluation/local-results/text-transformers/returned/` locally,
then run `.venv/Scripts/python.exe evaluation/merge_transformer_predictions.py` from the repo root.
The importer refuses missing/mismatching runs and writes evidence only to the ignored local
results folder. It does not select, deploy, edit reviewed reports or run sealed tests.


In [ ]:
archive = package_results(OUT, BUNDLE)
receipt = read_json(OUT / 'results_manifest.json')
print('Completed runs:', len(receipt['files']), '/ 6. Complete:', receipt['complete'])
files.download(str(archive))
